# Reproducing and Extending SimCSE
COMP8240 Applications of Data Science, Macquarie University, Session 2 2026 — Trilok Kumar Reddy Ande

Run the cells from top to bottom on a GPU runtime (Google Colab, T4). See `README.md` for details.

## 0. Setup

In [ ]:
# Paths: everything is saved in a local "data" folder next to this notebook.
# Set USE_DRIVE = True to save to Google Drive instead (Colab only).
import os
USE_DRIVE = False
ROOT = os.getcwd()
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/COMP8240_WikiSim"
else:
    SAVE_DIR = os.path.join(ROOT, "data")
os.makedirs(SAVE_DIR, exist_ok=True)
print("Saving data to", SAVE_DIR)

In [ ]:
import sys, torch, transformers
print("Python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 1. Replication of the original results
Clone the official SimCSE code. The `sed` line fixes `batch_encode_plus`, which was removed from newer versions of `transformers`.

In [ ]:
!git clone https://github.com/princeton-nlp/SimCSE.git
%cd SimCSE
!pip install -q transformers scipy scikit-learn prettytable
!sed -i "s/tokenizer.batch_encode_plus(/tokenizer(/g" evaluation.py

In [ ]:
%cd SentEval/data/downstream/
!bash download_dataset.sh
%cd /content/SimCSE

### 1.1 Supervised SimCSE (BERT-base)

In [ ]:
!python evaluation.py \
  --model_name_or_path princeton-nlp/sup-simcse-bert-base-uncased \
  --pooler cls --task_set sts --mode test

### 1.2 Unsupervised SimCSE (BERT-base)

In [ ]:
!python evaluation.py \
  --model_name_or_path princeton-nlp/unsup-simcse-bert-base-uncased \
  --pooler cls_before_pooler --task_set sts --mode test

### 1.3 Sanity check

In [ ]:
from simcse import SimCSE

model = SimCSE("princeton-nlp/sup-simcse-bert-base-uncased")

query = ["Two dogs are running."]
others = [
    "There are animals outdoors.",       # should be similar (entailment)
    "The pets are sitting on a couch.",  # should be less similar (contradiction)
    "The stock market fell today.",      # should be very different (unrelated)
]

scores = model.similarity(query, others)[0]
for sentence, score in zip(others, scores):
    print(f"{score:.3f}  {sentence}")

print()
print("Tricky pair (same words, different meaning):")
print(f"{model.similarity('The dog bit the man.', 'The man bit the dog.'):.3f}")

## 2. New existing data: PAWS

In [ ]:
!pip install -q datasets
from datasets import load_dataset

paws = load_dataset("google-research-datasets/paws", "labeled_final", split="test")

print("Number of pairs:", len(paws))
print("Columns:", paws.column_names)
print("Label counts:", {0: paws["label"].count(0), 1: paws["label"].count(1)})
print()
for i in range(3):
    print("Sentence 1:", paws[i]["sentence1"])
    print("Sentence 2:", paws[i]["sentence2"])
    print("Label:", paws[i]["label"], "(1 = same meaning, 0 = different meaning)")
    print()

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

s1 = paws["sentence1"]
s2 = paws["sentence2"]
labels = np.array(paws["label"])

def simcse_scores(name):
    m = SimCSE(name)
    a = m.encode(s1, batch_size=128)
    b = m.encode(s2, batch_size=128)
    return (a * b).sum(dim=1).cpu().numpy()   # cosine similarity for each pair

def overlap_scores():
    out = []
    for x, y in zip(s1, s2):
        A, B = set(x.lower().split()), set(y.lower().split())
        out.append(len(A & B) / len(A | B))     # share of words in common
    return np.array(out)

paws_results = {
    "Word overlap (baseline)": overlap_scores(),
    "Unsup SimCSE": simcse_scores("princeton-nlp/unsup-simcse-bert-base-uncased"),
    "Sup SimCSE": simcse_scores("princeton-nlp/sup-simcse-bert-base-uncased"),
}

print()
for name, sc in paws_results.items():
    auc = roc_auc_score(labels, sc)
    print(f"{name:25s} ROC-AUC = {auc:.3f} | avg score: same meaning = {sc[labels==1].mean():.3f}, different = {sc[labels==0].mean():.3f}")

### 2.1 Highest-scoring different-meaning pairs

In [ ]:
sc = paws_results["Sup SimCSE"]
different = np.where(labels == 0)[0]                     # only different-meaning pairs
worst = different[np.argsort(-sc[different])[:3]]        # the 3 highest scores among them

for i in worst:
    print(f"Score: {sc[i]:.3f}   (true label: 0 = different meaning)")
    print("  1:", s1[i])
    print("  2:", s2[i])
    print()

### 2.2 Identical pairs and real mistakes

In [ ]:
identical = np.array([a.strip() == b.strip() for a, b in zip(s1, s2)])
print("Identical pairs:", identical.sum(), "out of", len(labels))
print("  labelled 0 (different):", (identical & (labels == 0)).sum())
print("  labelled 1 (same):     ", (identical & (labels == 1)).sum())
print()

different_real = np.where((labels == 0) & ~identical)[0]
worst = different_real[np.argsort(-sc[different_real])[:3]]

print("Worst real mistakes (non-identical pairs):\n")
for i in worst:
    print(f"Score: {sc[i]:.3f}   (true label: 0 = different meaning)")
    print("  1:", s1[i])
    print("  2:", s2[i])
    print()

## 3. New constructed data: WikiSim
### 3.1 Download 20 Wikipedia articles (with retries and revision IDs)

In [ ]:
import requests, json, time

TOPICS = {
    "animals":   ["Koala", "Kangaroo", "Platypus", "Wombat"],
    "planets":   ["Mars", "Jupiter", "Venus", "Saturn"],
    "sports":    ["Cricket", "Rugby union", "Tennis", "Association football"],
    "buildings": ["Sydney Opera House", "Eiffel Tower", "Taj Mahal", "Colosseum"],
    "diseases":  ["Malaria", "Tuberculosis", "Influenza", "Measles"],
}

HEADERS = {"User-Agent": "COMP8240-SimCSE-replication (student project; https://github.com/Trilok119/comp8240-simcse-replication)"}

def fetch(title, tries=5):
    for attempt in range(tries):
        r = requests.get(
            "https://en.wikipedia.org/w/api.php",
            params={"action": "query", "prop": "extracts|revisions", "explaintext": 1,
                    "exsectionformat": "wiki", "rvprop": "ids|timestamp",
                    "titles": title, "redirects": 1, "format": "json"},
            headers=HEADERS,
        )
        try:
            page = next(iter(r.json()["query"]["pages"].values()))
            return page["extract"], page["revisions"][0]["revid"], page["revisions"][0]["timestamp"]
        except Exception:
            wait = 5 * (attempt + 1)
            print(f"   ...empty reply for {title} (status {r.status_code}), waiting {wait}s and retrying")
            time.sleep(wait)
    raise RuntimeError(f"Could not download {title}")

# If raw_articles.json is already in the data folder (it is in the GitHub repo),
# load it so the exact same Wikipedia revisions are used. Otherwise download.
RAW_PATH = f"{SAVE_DIR}/raw_articles.json"
raw = json.load(open(RAW_PATH)) if os.path.exists(RAW_PATH) else {}
print("Articles already saved:", len(raw))

for topic, titles in TOPICS.items():
    for t in titles:
        if t in raw:
            continue                      # already downloaded
        text, revid, ts = fetch(t)
        raw[t] = {"topic": topic, "text": text, "revid": revid, "timestamp": ts}
        print(f"{topic:10s} {t:22s} {len(text.split()):6d} words   (revision {revid})")
        time.sleep(1)                     # be polite to Wikipedia

with open(RAW_PATH, "w") as f:
    json.dump(raw, f)

print("\nSaved", len(raw), "articles to", SAVE_DIR)

### 3.2 Clean and split into sentences

In [ ]:
import re, random, nltk
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize

SKIP = {"see also", "references", "external links", "further reading", "notes",
        "bibliography", "sources", "citations", "gallery", "footnotes"}

records = []
stats = {"kept": 0, "too_short": 0, "too_long": 0, "junk": 0}

for title, art in raw.items():
    section, para_id = "Introduction", 0
    for line in art["text"].split("\n"):
        line = line.strip()
        if not line:
            continue
        heading = re.match(r"^(=+)\s*(.*?)\s*=+$", line)
        if heading:
            if len(heading.group(1)) == 2:          # main section heading (== Name ==)
                section = heading.group(2)
            continue
        if section.lower() in SKIP:
            continue
        para_id += 1
        for pos, s in enumerate(sent_tokenize(line)):
            n = len(s.split())
            if n < 8:
                stats["too_short"] += 1; continue
            if n > 40:
                stats["too_long"] += 1; continue
            if any(c in s for c in "{}\\") or not re.search("[a-zA-Z]", s):
                stats["junk"] += 1; continue
            records.append({"article": title, "topic": art["topic"], "section": section,
                            "para": para_id, "pos": pos, "text": s})
            stats["kept"] += 1

print("Cleaning summary:", stats)
print("Total clean sentences:", len(records))
print()
for title in raw:
    rows = [r for r in records if r["article"] == title]
    print(f"{title:22s} {len(rows):4d} sentences, {len({r['section'] for r in rows}):2d} sections")

print("\nThree random examples:")
random.seed(42)
for r in random.sample(records, 3):
    print(f"  [{r['topic']} | {r['article']} | {r['section']}] {r['text']}")

with open(f"{SAVE_DIR}/sentences.json", "w") as f:
    json.dump(records, f)
print("\nSaved to", SAVE_DIR)

### 3.3 Build 400 pairs at four relatedness levels

In [ ]:
from collections import defaultdict
import pandas as pd

random.seed(42)
N = 100

# Level 3: neighbouring sentences in the same paragraph
by_para = defaultdict(list)
for i, r in enumerate(records):
    by_para[(r["article"], r["para"])].append(i)
adjacent = [(a, b) for idx in by_para.values() for a, b in zip(idx, idx[1:])
            if records[b]["pos"] - records[a]["pos"] == 1]
level3 = random.sample(adjacent, N)

# Levels 2, 1, 0: random pairs that follow each rule
def pick(rule, n):
    chosen = set()
    while len(chosen) < n:
        a, b = random.sample(range(len(records)), 2)
        if rule(records[a], records[b]):
            chosen.add((min(a, b), max(a, b)))
    return list(chosen)

level2 = pick(lambda x, y: x["article"] == y["article"] and x["section"] != y["section"], N)
level1 = pick(lambda x, y: x["topic"] == y["topic"] and x["article"] != y["article"], N)
level0 = pick(lambda x, y: x["topic"] != y["topic"], N)

rows = []
for level, pairs in [(3, level3), (2, level2), (1, level1), (0, level0)]:
    for a, b in pairs:
        A, B = records[a], records[b]
        rows.append({"level": level, "s1": A["text"], "s2": B["text"],
                     "article1": A["article"], "article2": B["article"],
                     "section1": A["section"], "section2": B["section"]})

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
df.insert(0, "pair_id", range(1, len(df) + 1))

def overlap(x, y):
    A, B = set(x.lower().split()), set(y.lower().split())
    return len(A & B) / len(A | B)
df["word_overlap"] = [overlap(x, y) for x, y in zip(df.s1, df.s2)]

df.to_csv(f"{SAVE_DIR}/wikisim_pairs.csv", index=False)

print("Pairs per level:", df.level.value_counts().sort_index().to_dict())
print("Neighbouring-sentence candidates available:", len(adjacent))
print("\nAverage word overlap per level:")
print(df.groupby("level").word_overlap.mean().round(3).to_dict())

print("\nOne example per level:")
for lv in [3, 2, 1, 0]:
    r = df[df.level == lv].iloc[0]
    print(f"\nLevel {lv}: [{r.article1} / {r.section1}]  vs  [{r.article2} / {r.section2}]")
    print("  1:", r.s1)
    print("  2:", r.s2)

print("\nSaved", len(df), "pairs to", SAVE_DIR)

### 3.4 Score WikiSim with SimCSE

In [ ]:
from scipy.stats import spearmanr

def encode_pairs(name, a_list, b_list):
    m = SimCSE(name)
    a = m.encode(a_list, batch_size=128)
    b = m.encode(b_list, batch_size=128)
    return (a * b).sum(dim=1).cpu().numpy()

s1_list, s2_list = df.s1.tolist(), df.s2.tolist()
df["unsup_simcse"] = encode_pairs("princeton-nlp/unsup-simcse-bert-base-uncased", s1_list, s2_list)
df["sup_simcse"]   = encode_pairs("princeton-nlp/sup-simcse-bert-base-uncased", s1_list, s2_list)

print()
for col, name in [("word_overlap", "Word overlap (baseline)"),
                  ("unsup_simcse", "Unsup SimCSE"),
                  ("sup_simcse", "Sup SimCSE")]:
    rho = spearmanr(df.level, df[col]).correlation
    print(f"{name:25s} Spearman = {rho * 100:.2f}")

print("\nAverage similarity score per level:")
print(df.groupby("level")[["word_overlap", "unsup_simcse", "sup_simcse"]].mean().round(3))

df.to_csv(f"{SAVE_DIR}/wikisim_pairs.csv", index=False)
print("\nSaved scores to", SAVE_DIR)

### 3.5 Create the annotation sheet (100 pairs, levels hidden)

In [ ]:
ann_sample = pd.concat([df[df.level == lv].sample(25, random_state=7) for lv in [0, 1, 2, 3]])
ann = ann_sample[["pair_id", "s1", "s2"]].sample(frac=1, random_state=7).reset_index(drop=True)
ann["human_score"] = ""

ann.to_csv(f"{SAVE_DIR}/annotation_sheet.csv", index=False)
print("Created annotation sheet with", len(ann), "pairs")
print("Levels in sample:", ann_sample.level.value_counts().sort_index().to_dict())
print("\nFirst 3 rows:")
print(ann.head(3).to_string())

### 3.6 Check for broken sentences

In [ ]:
def is_broken(s):
    unbalanced = s.count("(") != s.count(")")
    bad_end = re.search(r"\b(i\.e|e\.g|c|ca|approx|St|Dr|Mt|vs|No)\.$", s) is not None
    return unbalanced or bad_end

broken_sents = [r["text"] for r in records if is_broken(r["text"])]
pair_broken = df.s1.apply(is_broken) | df.s2.apply(is_broken)
ann_broken = ann.s1.apply(is_broken) | ann.s2.apply(is_broken)

print(f"Broken sentences: {len(broken_sents)} of {len(records)} ({100*len(broken_sents)/len(records):.1f}%)")
print(f"Pairs with a broken sentence: {pair_broken.sum()} of {len(df)}")
print(f"Annotation pairs with a broken sentence: {ann_broken.sum()} of {len(ann)}")
print("\nFive examples:")
for s in broken_sents[:5]:
    print("  -", s)

## 4. Human annotation
`annotation_done.csv` holds the author's 0–5 scores for the 100 pairs.

In [ ]:
human = pd.read_csv(f"{SAVE_DIR}/annotation_done.csv")[["pair_id", "human_score"]]
merged = df.merge(human, on="pair_id")
print("Annotated pairs found:", len(merged))

print("\n1) Do my scores agree with the automatic levels?")
print(f"   Spearman (human vs level) = {spearmanr(merged.level, merged.human_score).correlation * 100:.2f}")
print("   My average score per level:", merged.groupby("level").human_score.mean().round(2).to_dict())

print("\n2) How well does each method match MY scores?")
for col, name in [("word_overlap", "Word overlap"), ("unsup_simcse", "Unsup SimCSE"), ("sup_simcse", "Sup SimCSE")]:
    vs_human = spearmanr(merged.human_score, merged[col]).correlation * 100
    vs_level = spearmanr(merged.level, merged[col]).correlation * 100
    print(f"   {name:14s} vs my scores = {vs_human:.2f}   |   vs levels (same 100 pairs) = {vs_level:.2f}")

merged.to_csv(f"{SAVE_DIR}/annotated_100.csv", index=False)
print("\nSaved to", SAVE_DIR)

### 4.1 Agreement with a second (AI) annotator

In [ ]:
# Agreement between the author's scores and a second annotator.
# NOTE: the second annotator is an AI model (Claude, Anthropic), not a person.
from sklearn.metrics import cohen_kappa_score
a = pd.read_csv(f"{SAVE_DIR}/annotation_done.csv")[["pair_id", "human_score"]].rename(columns={"human_score": "author"})
b = pd.read_csv(f"{SAVE_DIR}/annotation_claude.csv")[["pair_id", "human_score"]].rename(columns={"human_score": "ai_annotator"})
both = a.merge(b, on="pair_id")
print("Pairs:", len(both))
print(f"Spearman                 = {spearmanr(both.author, both.ai_annotator).correlation * 100:.2f}")
print(f"Quadratic weighted kappa = {cohen_kappa_score(both.author, both.ai_annotator, weights='quadratic'):.3f}")
print(f"Exact agreement          = {(both.author == both.ai_annotator).mean():.0%}")
print(f"Within 1 point           = {((both.author - both.ai_annotator).abs() <= 1).mean():.0%}")

## 5. LLM as a judge (Qwen2.5-3B-Instruct)
### 5.1 Version 1 prompt

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.float16).to("cuda")

GUIDE = """You rate how related two sentences are, on a scale from 0 to 5.
5 = same specific point
4 = same subject, closely connected details
3 = same subject, different aspect
2 = same broad area, different subject (e.g. koala vs kangaroo)
1 = very loosely connected
0 = unrelated
Reply with a single digit only."""

def judge(a, b):
    msgs = [{"role": "system", "content": GUIDE},
            {"role": "user", "content": f"Sentence 1: {a}\nSentence 2: {b}\nScore:"}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to("cuda")
    out = llm.generate(**ids, max_new_tokens=3, do_sample=False)
    reply = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"[0-5]", reply)
    return int(m.group()) if m else None

merged["llm_score"] = [judge(a, b) for a, b in zip(merged.s1, merged.s2)]

ok = merged.llm_score.notna()
print("Pairs the AI scored:", ok.sum(), "of", len(merged))
print("AI score counts:", merged.llm_score.value_counts().sort_index().to_dict())
print(f"\nAI vs my scores: Spearman = {spearmanr(merged.human_score[ok], merged.llm_score[ok]).correlation * 100:.2f}")
print(f"AI vs levels:    Spearman = {spearmanr(merged.level[ok], merged.llm_score[ok]).correlation * 100:.2f}")
print("AI average per level:", merged[ok].groupby("level").llm_score.mean().round(2).to_dict())

merged.to_csv(f"{SAVE_DIR}/annotated_100.csv", index=False)
print("\nSaved to", SAVE_DIR)

### 5.2 Version 2 prompt

In [ ]:
GUIDE_V2 = """You rate how related two sentences are, on a scale from 0 to 5.
Two sentences can be related even if they share no words: judge whether they are about
the same thing, the same kind of thing, or nothing in common.

5 = same specific point (e.g. both about the opening hours of the same building)
4 = same subject, closely connected details (e.g. a building's entry gates and its opening hours)
3 = same subject, different aspect (e.g. a disease's symptoms and its prevention)
2 = same broad area, different subject (e.g. one about koalas, one about kangaroos)
1 = very loosely connected (e.g. both just mention the same country or a date)
0 = unrelated (e.g. one about kangaroos, one about virus genetics)
Reply with a single digit only."""

def judge_v2(a, b):
    msgs = [{"role": "system", "content": GUIDE_V2},
            {"role": "user", "content": f"Sentence 1: {a}\nSentence 2: {b}\nScore:"}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to("cuda")
    out = llm.generate(**ids, max_new_tokens=3, do_sample=False)
    reply = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"[0-5]", reply)
    return int(m.group()) if m else None

merged["llm_score_v2"] = [judge_v2(a, b) for a, b in zip(merged.s1, merged.s2)]

ok = merged.llm_score_v2.notna()
print("Pairs scored:", ok.sum(), "of", len(merged))
print("AI score counts (v2):", merged.llm_score_v2.value_counts().sort_index().to_dict())
print(f"\nAI v2 vs my scores: Spearman = {spearmanr(merged.human_score[ok], merged.llm_score_v2[ok]).correlation * 100:.2f}")
print(f"AI v2 vs levels:    Spearman = {spearmanr(merged.level[ok], merged.llm_score_v2[ok]).correlation * 100:.2f}")
print("AI v2 average per level:", merged[ok].groupby("level").llm_score_v2.mean().round(2).to_dict())

merged.to_csv(f"{SAVE_DIR}/annotated_100.csv", index=False)
print("\nSaved to", SAVE_DIR)
